In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del archivo language.csv

### Importar librerias

In [0]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType
from pyspark.sql.functions import lit, current_timestamp

### 1. Lectura del archivo

In [0]:
language_df = (
    spark.read
        .format("csv")
        .option("header", True)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/language.csv")
)

languageId,languageCode,languageName
24574,en,English
24575,sv,svenska
24576,de,Deutsch
24577,xx,unknown
24578,ja,Nihongo
24579,fr,Français
24580,es,Español
24581,ar,al-?arabiyyah
24582,la,Latin
24583,km,Khémôrôphéasa


### Cargue de archivo + Definir schema 

In [0]:
language_schema = StructType([
    StructField("languageId", IntegerType(), False),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
])

language_df = (
    spark.read
        .schema(language_schema)
        .format("csv")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/language.csv")
)

### 2. Eliminar columnas 

In [0]:
language_df = language_df.drop("languageCode")

languageId,languageName
null,languageName
24574,English
24575,svenska
24576,Deutsch
24577,unknown
24578,Nihongo
24579,Français
24580,Español
24581,al-?arabiyyah
24582,Latin


### 3. Renombrar columnas

In [0]:
language_df  = language_df.withColumnsRenamed({"languageId": "language_id", "languageName": "language_name"})

language_id,language_name
null,languageName
24574,English
24575,svenska
24576,Deutsch
24577,unknown
24578,Nihongo
24579,Français
24580,Español
24581,al-?arabiyyah
24582,Latin


### 4. Crear columnas

In [0]:
language_df  = (
    add_ingestion_date(language_df) 
    .withColumns({
        "env": lit(v_env),
        "file_date": lit(v_file_date)
    })
)

language_id,language_name,ingestion_date,env,file_date
null,languageName,2026-10-08T07:15:41.767Z,dev,2024-12-16
24574,English,2026-10-08T07:15:41.767Z,dev,2024-12-16
24575,svenska,2026-10-08T07:15:41.767Z,dev,2024-12-16
24576,Deutsch,2026-10-08T07:15:41.767Z,dev,2024-12-16
24577,unknown,2026-10-08T07:15:41.767Z,dev,2024-12-16
24578,Nihongo,2026-10-08T07:15:41.767Z,dev,2024-12-16
24579,Français,2026-10-08T07:15:41.767Z,dev,2024-12-16
24580,Español,2026-10-08T07:15:41.767Z,dev,2024-12-16
24581,al-?arabiyyah,2026-10-08T07:15:41.767Z,dev,2024-12-16
24582,Latin,2026-10-08T07:15:41.767Z,dev,2024-12-16


### 5. Escribir archivo

In [0]:
(
    language_df.write
        .mode("overwrite")
        .saveAsTable("movie_silver.languages")
)

In [0]:
dbutils.notebook.exit("Success")